# Localised LSTM fh=1 modified

This notebook reports `avg_normalised_loss`, `avg_actual_loss`, saves base/reconciled/naive forecasts, and evaluates MAE, MASE, RMSSE.

For LLSTM, each partition/local model applies early stopping independently using the same `args_EARLY_STOP_TOL` and `args_MIN_EPOCHS`. The aggregate loss table is for reporting only.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from llstm import (
    Datacollector,
    run_localised,
    compute_metrics_from_dicts,
    make_reconciliation_frames,
    evaluate_reconciliation_results,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
# -------------------------
# Parameter setup
# -------------------------
#args_path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/GEF2017_univarivate.csv'
args_path = 'E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv'
args_fh  = 5
args_lags = 48
args_INPUT, args_HIDDEN, args_LAYERS, args_DROP = 1, 64, 2, 0.1
args_EPOCHS, args_BATCH_SIZE, args_LR = 100 + args_fh*50, 256, 1e-3
args_TRAIN_RATIO = 0.8
args_MODEL_NAME = 'llstm'
args_truncated = None
args_DROP_BOUNDARY_GAP = True

# Early stopping settings
args_EARLY_STOP_ENABLED = True
args_EARLY_STOP_TOL = 1e-5
args_MIN_EPOCHS = 20 + args_fh*50

dict_ = Datacollector(pd.read_csv(args_path), lags=args_lags, ts=range(15), fh=args_fh)
df = pd.concat([dict_[key] for key in dict_.keys()]).dropna()
df['ds'] = pd.to_datetime(df['ds'], format='mixed')
if args_truncated is not None:
    df = df[df['ds'] >= args_truncated].reset_index(drop=True)
df = df.reset_index(drop=True)

parts = sorted(df['partition_id'].unique())
lag_cols_reversed = list(reversed(sorted([c for c in df.columns if c.startswith('lags_')], key=lambda x: int(x.split('_')[1]))))
forecast_horizon = ['y'] + sorted([c for c in df.columns if c.startswith('post_')], key=lambda x: int(x.split('_')[1]))
df2 = df[['unique_id', 'partition_id', 'ds'] + lag_cols_reversed + forecast_horizon].copy()

print(df2.head())
print(f'num partitions: {len(parts)}')
print(f'X shape: {df2[lag_cols_reversed].shape} | y shape: {df2[forecast_horizon].shape}')

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 17.07it/s]


  unique_id  partition_id                  ds  lags_48  lags_47  lags_46  \
0     Total             0 2003-03-03 00:00:00  12864.0  12389.0  12155.0   
1     Total             0 2003-03-03 01:00:00  12389.0  12155.0  12072.0   
2     Total             0 2003-03-03 02:00:00  12155.0  12072.0  12162.0   
3     Total             0 2003-03-03 03:00:00  12072.0  12162.0  12569.0   
4     Total             0 2003-03-03 04:00:00  12162.0  12569.0  13238.0   

   lags_45  lags_44  lags_43  lags_42  ...   lags_4   lags_3   lags_2  \
0  12072.0  12162.0  12569.0  13238.0  ...  15815.0  14745.0  13444.0   
1  12162.0  12569.0  13238.0  14191.0  ...  14745.0  13444.0  12350.0   
2  12569.0  13238.0  14191.0  15213.0  ...  13444.0  12350.0  11657.0   
3  13238.0  14191.0  15213.0  15646.0  ...  12350.0  11657.0  11357.0   
4  14191.0  15213.0  15646.0  15653.0  ...  11657.0  11357.0  11315.0   

    lags_1        y   post_1   post_2   post_3   post_4   post_5  
0  12350.0  11657.0  11357.0  11315.0

In [3]:
%%time
results = run_localised(
    df=df2,
    lag_cols_reversed=lag_cols_reversed,
    forecast_horizon=forecast_horizon,
    input_size=args_INPUT,
    hidden_size=args_HIDDEN,
    num_layers=args_LAYERS,
    dropout=args_DROP,
    batch_size=args_BATCH_SIZE,
    epochs=args_EPOCHS,
    lr=args_LR,
    train_ratio=args_TRAIN_RATIO,
    partition_col='partition_id',
    device=None,
    clip_grad=1.0,
    verbose=True,
    lags=args_lags,
    fh=args_fh,
    time_col='ds',
    drop_boundary_gap=args_DROP_BOUNDARY_GAP,
    early_stop_enabled=args_EARLY_STOP_ENABLED,
    early_stop_tol=args_EARLY_STOP_TOL,
    min_epochs=args_MIN_EPOCHS,
)

parts = results['parts']
train_idx = results['train_idx']
test_idx = results['test_idx']
dict_pred = results['dict_pred']
dict_true = results['dict_true']
dict_tr_pred = results['dict_tr_pred']
dict_tr_true = results['dict_tr_true']
dict_naive = results['dict_naive']

print('Device:', results['device'])
print('Stopped early:', results['stopped_early'])
print('Final avg_normalised_loss:', results['final_avg_normalised_loss'])
print('Final avg_actual_loss:', results['final_avg_actual_loss'])
print('Early stop rule: each local model checks its own avg_normalised_loss delta independently.')
local_stop_summary_df = pd.DataFrame(results['local_stop_summary'])
display(local_stop_summary_df.tail())
pd.DataFrame(results['round_logs']).tail()

[Localised] pid=0 epoch 001/350 | avg_normalised_loss=0.052912 | avg_actual_loss=443569.551169 | delta=nan | stopped=False
[Localised] pid=0 epoch 002/350 | avg_normalised_loss=0.037447 | avg_actual_loss=313919.052605 | delta=0.0154657 | stopped=False
[Localised] pid=0 epoch 003/350 | avg_normalised_loss=0.029965 | avg_actual_loss=251200.905285 | delta=0.00748152 | stopped=False
[Localised] pid=0 epoch 004/350 | avg_normalised_loss=0.028449 | avg_actual_loss=238491.262622 | delta=0.00151611 | stopped=False
[Localised] pid=0 epoch 005/350 | avg_normalised_loss=0.024075 | avg_actual_loss=201820.662408 | delta=0.00437436 | stopped=False
[Localised] pid=0 epoch 006/350 | avg_normalised_loss=0.022031 | avg_actual_loss=184688.911004 | delta=0.00204361 | stopped=False
[Localised] pid=0 epoch 007/350 | avg_normalised_loss=0.019527 | avg_actual_loss=163699.809302 | delta=0.00250375 | stopped=False
[Localised] pid=0 epoch 008/350 | avg_normalised_loss=0.019504 | avg_actual_loss=163504.240426 | d

,partition_id,stopped_early,stop_epoch,stop_reason,early_stop_tol,min_epochs,final_avg_normalised_loss,final_avg_actual_loss,n_train,n_test
5,5,False,350,max_epochs,0.00001,270,0.005377,226.372249,99283,24830
6,6,True,274,normalised_loss_delta<1e-05 after min_epochs=270,0.00001,270,0.006504,86.022951,99283,24830
7,7,True,319,normalised_loss_delta<1e-05 after min_epochs=270,0.00001,270,0.005246,1691.080554,99283,24830
8,8,True,298,normalised_loss_delta<1e-05 after min_epochs=270,0.00001,270,0.004965,755.840803,99283,24830
9,9,True,279,normalised_loss_delta<1e-05 after min_epochs=270,0.00001,270,0.005553,864.335929,99283,24830


CPU times: total: 8h 55min 33s
Wall time: 3h 28min 3s


,epoch,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,n_active_partitions_reported,n_partitions_stopped_this_epoch
345,346,0.005143,216.497376,0.000345,1,0
346,347,0.005251,221.053292,0.000108,1,0
347,348,0.005399,227.286668,0.000148,1,0
348,349,0.005193,218.633787,0.000206,1,0
349,350,0.005377,226.372249,0.000184,1,0


In [4]:
# Base model and Naive baseline metrics before reconciliation
base_metrics = compute_metrics_from_dicts(dict_true, dict_pred, dict_train_true=dict_tr_true, h_idx=0)
naive_metrics = compute_metrics_from_dicts(dict_true, dict_naive, dict_train_true=dict_tr_true, h_idx=0)
display(base_metrics.tail())
display(naive_metrics.tail())

,partition_id,n_test,MAE,MASE,RMSSE
6,6,24830,10.531998,0.427127,0.465160
7,7,24830,25.901637,0.242757,0.253506
8,8,24830,22.127583,0.286609,0.296263
9,9,24830,19.388915,0.248916,0.276012
10,Overall,248300,29.958354,0.248739,0.261246


,partition_id,n_test,MAE,MASE,RMSSE
6,6,24830,23.280522,0.944147,0.890569
7,7,24830,99.788858,0.935247,0.924315
8,8,24830,70.846740,0.917648,0.888023
9,9,24830,70.184900,0.901039,0.876585
10,Overall,248300,130.509368,0.920088,0.898420


In [5]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

#path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/'
path = 'E:/yjz/Datasets/GEFCOM2017/'
S = pd.read_csv(f'{path}GEF_S.csv')
tags = pd.read_pickle(f'{path}tags.pkl')
def align_S_for_hierarchicalforecast(S, id_col="unique_id"):
    S = S.copy()
    if id_col not in S.columns:
        S = S.reset_index()
        if id_col not in S.columns:
            S = S.rename(columns={S.columns[0]: id_col})
    bottom_ids = [c for c in S.columns if c != id_col]
    agg_ids = [uid for uid in S[id_col].tolist() if uid not in bottom_ids]
    row_order = agg_ids + bottom_ids
    S_aligned = (
        S
        .set_index(id_col)
        .loc[row_order, bottom_ids]
        .reset_index())
    bottom_block = S_aligned[bottom_ids].tail(len(bottom_ids)).to_numpy()
    if not np.allclose(bottom_block, np.eye(len(bottom_ids))):
        raise ValueError("S Error")
    return S_aligned

S = align_S_for_hierarchicalforecast(S)
reconciliation_methods = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
    MinTrace(method='wls_var'),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
]
reconcilers = HierarchicalReconciliation(reconcilers=reconciliation_methods)

recon_results = {}
train_frames = {}
test_frames = {}
for H_IDX in range(args_fh + 1):
    y_df, y_hat_df, train_frame, test_frame = make_reconciliation_frames(
        df=df2,
        train_idx=train_idx,
        test_idx=test_idx,
        parts=parts,
        dict_tr_pred=dict_tr_pred,
        dict_pred=dict_pred,
        forecast_horizon=forecast_horizon,
        base_model_name=args_MODEL_NAME,
        horizon_idx=H_IDX,
    )
    train_frames[H_IDX] = train_frame
    test_frames[H_IDX] = test_frame
    recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S_df=S, tags=tags, Y_df=y_df)

recon_results[0].head()

,unique_id,ds,y,llstm,llstm/BottomUp,llstm/MinTrace_method-mint_shrink,llstm/MinTrace_method-wls_var,llstm/MinTrace_method-ols,llstm/MinTrace_method-wls_struct
0,Total,2014-06-30 23:00:00,15138.0,15218.693359,15187.237183,15174.525598,15176.096329,15206.871076,15183.556439
1,Total,2014-07-01 00:00:00,13810.0,13914.613281,13787.446899,13809.916853,13809.659683,13895.156508,13842.919888
2,Total,2014-07-01 01:00:00,12979.0,13161.219727,12989.747437,13003.242490,13004.398907,13128.537141,13048.541622
3,Total,2014-07-01 02:00:00,12468.0,12516.145508,12457.171326,12451.332796,12452.668008,12500.884327,12467.409853
4,Total,2014-07-01 03:00:00,12226.0,12358.377930,12279.493164,12265.000530,12267.384359,12335.400363,12286.822636


In [6]:
# Save base, reconciled and naive forecasts; evaluate MAE, MASE and RMSSE
output_prefix = f'fh{args_fh+1}_loc'
eval_outputs = evaluate_reconciliation_results(
    recon_results=recon_results,
    train_frames=train_frames,
    test_frames=test_frames,
    forecast_horizon=forecast_horizon,
    base_model_name=args_MODEL_NAME,
    output_prefix=output_prefix,
    approach='localised',
    round_logs=results['round_logs'],
    timings=results['timings'],
)

forecast_table = eval_outputs['forecast_table']
per_series_metrics = eval_outputs['per_series_metrics']
metrics_by_level = eval_outputs['metrics_by_level']
overall_metrics = eval_outputs['overall_metrics']
timing_df = eval_outputs['timing_df']
output_paths = eval_outputs['output_paths']

# LLSTM-specific early stopping diagnostics: each local model has its own stop epoch.
local_stop_summary_path = f'{output_prefix}_local_stop_summary.csv'
round_logs_by_partition_path = f'{output_prefix}_round_logs_by_partition.csv'
pd.DataFrame(results['local_stop_summary']).to_csv(local_stop_summary_path, index=False)
pd.DataFrame(results['round_logs_by_partition']).to_csv(round_logs_by_partition_path, index=False)
output_paths['local_stop_summary'] = local_stop_summary_path
output_paths['round_logs_by_partition'] = round_logs_by_partition_path

print('Saved outputs:')
for k, v in output_paths.items():
    print(f'  {k}: {v}')

metrics_by_level

Saved outputs:
  forecasts: fh6_loc_forecasts.csv
  per_series_metrics: fh6_loc_per_series_metrics.csv
  metrics_by_level: fh6_loc_metrics_by_level.csv
  overall_metrics: fh6_loc_overall_metrics.csv
  round_logs: fh6_loc_round_logs.csv
  timing: fh6_loc_timing.csv
  local_stop_summary: fh6_loc_local_stop_summary.csv
  round_logs_by_partition: fh6_loc_round_logs_by_partition.csv


,level,role,method,MAE,MASE,RMSSE,n_series,approach
0,1,top,base,238.295686,0.411912,0.438442,1,localised
1,1,top,bu,229.471423,0.396659,0.412709,1,localised
2,1,top,mint_ols,230.172636,0.397871,0.422819,1,localised
3,1,top,mint_shrinkage,224.224063,0.387588,0.404702,1,localised
4,1,top,mint_var,221.668028,0.383170,0.401275,1,localised
5,1,top,naive,1632.296309,2.821548,2.639286,1,localised
6,1,top,wls_structure,218.984890,0.378532,0.399293,1,localised
7,2,middle,base,50.617397,0.555504,0.588815,6,localised
8,2,middle,bu,48.956737,0.549121,0.580078,6,localised
9,2,middle,mint_ols,53.797650,0.685496,0.718433,6,localised


In [7]:
overall_metrics

,method,MAE,MASE,RMSSE,n_series,approach
0,base,69.503167,0.551298,0.584869,10,localised
1,bu,67.624345,0.545943,0.577053,10,localised
2,mint_ols,70.619153,0.628518,0.662456,10,localised
3,mint_shrinkage,66.279500,0.539276,0.570871,10,localised
4,mint_var,65.755103,0.536617,0.568552,10,localised
5,naive,402.402792,2.813179,2.624031,10,localised
6,wls_structure,66.067028,0.550780,0.581426,10,localised


In [8]:
timing_df

,module,seconds
0,training_sec,12482.058432
1,total_sec,12483.226764
